# Create the inference wrapper

In order to use the model, we need to write a `PythonComponent` wrapper script that loads the model and feeds it by extracting the image from an `ImageSet` input payload, and prepare the output payload from the result of the segmentation. The `process_input` method will serve as an entrypoint for the wrapper.

An example script is already implemented under the `src` folder and in the `segmentation.py` file ([see here](../src/segmentation.py)). This notebook is purely for demonstrating how to write such a wrapper. Execution of this notebook is not expected, and changes being made in this tutorial will not be reflected in the final package.

## Load the model

The wrapper script must load the model first.

In [ ]:
from ultralytics import YOLO

model = YOLO(f"../models/yolo11n-seg.pt")

## Create ImageSet payload format

The method defined below creates the required format, and will be put into the payload with name `vision_payload` which will be provided by the AI Inference Server when an image is arrived from the selected camera through the `Vision Connector Application`.

In [ ]:
import cv2
import datetime

def create_imageset_dict(image_path):
    timestamp = datetime.datetime.now()

    image = cv2.imread(image_path)
    (height, width) = image.shape[:2]
    image_bytes = image.ravel().tobytes()

    return {
        "version": "1",  # version of the Metadata format
        "count": 1,  # Number of images on message
        "timestamp": timestamp.isoformat(),  # Camera acquisition time
        "detail": [{  # list of images with detailed information
            "id": str(image_path),  # unique image identifier. this case we use the filename of the original image
            "timestamp": str(timestamp.timestamp()),  # Timestamp provided by the camera
            "width": width,  # image width
            "height": height,  # image height
            "format": "BGR8",  # image format configure
            "metadata": "",  # optional extra information on image
            "image": bytes(image_bytes)  # image binary with the given 'format'
        }]
    }

input_payload = {"vision_payload": create_imageset_dict('../images/bus.jpg')}
input_payload

## Extract image from ImageSet

The wrapper script have to extract the image data from the payload, and create a BGR image for the model to process.

The original image is packaged into the payload in raw byte form, so the script has to convert it into a numpy array of the right shape.

In [ ]:
import numpy

image_set = input_payload["vision_payload"]
image_detail = image_set['detail'][0]
width = image_detail.get("width")
height = image_detail.get("height")
print(f"Original image width: {width}, height: {height}")
image_data = numpy.frombuffer(image_detail['image'], dtype=numpy.uint8)  # BGR (height x width x 3)
print(f"Image data shape: {image_data.shape}")
image_data = image_data.reshape(height, width, 3)                        # BGR (height, width, 3)
print(f"Image data shape: {image_data.shape}")
print(f"Image data type: {image_data.dtype}")

## Inferencing the model

In [ ]:
result = model(image_data)
result = result[0]  # Get the first (and only) result from the list

## Postprocessing

The end goal of the wrapper is to create an output payload with the all the information we need. We plan to include the following data in the result:

- the id of the image (its path and name),
- the detected classes with their calculated relative areas, as shown in notebook [10-UltralyticsYoloModel](./10-UltralyticsYoloModel.ipynb),
- the annotated summary image to visualize (if visualization is requested).

The id of the image is simply acquired from the input payload.

In [ ]:
output_payload = {}
output_payload["iuid"] = image_detail.get('id', 'no-id')

output_payload

Let's create a helper function that gather all area-related information from a segmentation result into a dictionary.

In [ ]:
def calculate_areas(result):
    areas = []

    class_ids = result.boxes.cls.int().tolist()
    masks = result.masks.data

    for class_id, mask in zip(class_ids, masks):
        class_label = result.names[class_id]
        mask_area = mask.sum().item()
        total_area = mask.numel()
        relative_area = mask_area / total_area
        areas.append({"class": class_label,
                      "relative_area": relative_area,
                      "text": f"Detected {class_label} occupying {relative_area * 100.0:.1f}% of the image."})
    return areas

calculate_areas(result)

If we want to add a dictionary to the output payload, we have to convert it into a `json` string. 

In [ ]:
import json
output_payload["areas"] = json.dumps(calculate_areas(result))

Finally, we want to put the annotated image provided by the YOLO model into the output payload, so that we can inspect it on the AI Inference Server. However, we only want to do this if visualization from the AI IS side is specifically requested.

When visualization is required, AI IS updates a boolean parameter called `__AI_IS_IMAGE_SET_VISUALIZATION` on the wrapper. We have to prepare out script by implementing the `update_parameters` function.

In [ ]:
__AI_IS_IMAGE_SET_VISUALIZATION = False  # By default we do not visualize the annotated image

def update_parameters(parameters: dict):
    global __AI_IS_IMAGE_SET_VISUALIZATION 
    __AI_IS_IMAGE_SET_VISUALIZATION = parameters.get("__AI_IS_IMAGE_SET_VISUALIZATION", __AI_IS_IMAGE_SET_VISUALIZATION)

If the visualization parameter is set to true, we put the annotated image into the output payload in an ImageSet format. To do that, we reuse the ImageSet of the input payload, and replace the raw image data with the annotated image.

In [ ]:
update_parameters({"__AI_IS_IMAGE_SET_VISUALIZATION": True})  # Enable visualization; this is called by AI IS when visualization is requested

if __AI_IS_IMAGE_SET_VISUALIZATION is True:
    result_img = result.plot()
    # Replace the original image with the result image in ImageSet
    image_set['detail'][0]['image'] = result_img.ravel().tobytes()
    output_payload["result_image_set"] = image_set

output_payload

With that, our output payload is ready. You can find the entire code for the wrapper in the [segmentation.py](../src/segmentation.py) code.

To make the script easier to read and to make the tutorial more general, we created an additional script, [imageset.py](../src/imageset.py), which contains a class `ImageSet` that handles the general processing of ImageSet formats. The class can read and prepare vision payload dictionaries, change the stored images, query the images in RGB format, and so on.

We will include this file among the resources when we create the pipeline in notebook [30-CreatePipeline](./30-CreatePipeline.ipynb).

## Ultralytics unsupported dependency workaround

Unfortunately, Ultralytics has unsupported dependencies which cannot be used on AI Inference Server at the time of writing. These dependencies are:

- `pytorch`, which is able to use CPU and GPU as well, but GPU is not presented for PythonComponent on AI Inference Server, and
- `opencv-python`, which depends on libGL library and that is not presented on AI Inference Server.

AI SDK can detect `pytorch` and automatically replaces it with a CPU only version. However, replacing `opencv-python` with `opencv-python-headless`, which is the same package but without GUI toolkit dependencies, is not that easy.

Nonetheless, AI SDK provides a workaround. __Note that this only works with AI SDK version 2.6 and above, on AI Inference Server version 2.6 or above.__

First, we create a `requirements.txt` ([see here](../src/requirements.txt)) with all the direct and transitive dependencies of Ultralytics, as well as every other packages and dependencies we need for our wrapper. One way to do this is to run a pip dry-run install command, and gather all the package names and versions pip would install for our wrapper.

There, we swap the `opencv-python` dependency to `opencv-python-headless` of the same version.

```python
#################################
# Direct dependencies
#################################
ultralytics==8.3.169

#################################
# Transitive dependencies
#################################
certifi==2025.7.14
charset-normalizer==3.4.2
contourpy==1.3.3
cycler==0.12.1
filelock==3.19.1
fonttools==4.59.1
fsspec==2025.7.0
idna==3.10
jinja2==3.1.6
kiwisolver==1.4.9
markupsafe==3.0.2
matplotlib==3.10.3
mpmath==1.3.0
networkx==3.5
numpy==2.2.6
# opencv-python is not supported on AI Inference Server; instead we use the headless version
# opencv-python==4.12.0.88
opencv-python-headless==4.12.0.88
packaging==25.0
pandas==2.3.1
pillow==11.3.0
psutil==7.0.0
py-cpuinfo==9.0.0
pyparsing==3.2.3
python-dateutil==2.9.0.post0
pytz==2025.2
pyyaml==6.0.2
requests==2.32.4
scipy==1.16.1
six==1.17.0
sympy==1.14.0
torch@https://download.pytorch.org/whl/cpu-cxx11-abi/torch-2.7.1%2Bcpu.cxx11.abi-cp312-cp312-linux_x86_64.whl#sha256=d06d422ac9bf250bbfc9b96921fc79354cc89f5ce6e5f98446fc4e90a66a23fb
torchvision@https://download.pytorch.org/whl/cpu/torchvision-0.22.1%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl#sha256=b5fa7044bd82c6358e8229351c98070cf3a7bf4a6e89ea46352ae6c65745ef94
tqdm==4.67.1
typing-extensions==4.14.1
tzdata==2025.2
ultralytics-thop==2.0.15
urllib3==2.5.0
```

When we create our wrapper, as seen in the [next notebook](./30-CreatePipeline.ipynb), we set the package requirements by providing the requirements.txt with `no_deps=True` parameter:

```python
component.set_requirements("../src/requirements.txt", no_deps=True)
```

This will tell AI SDK and AI IS to download and install all the packages without any of their transitive dependencies; hence, Ultralytics will use the headless version of opencv-python.

__Keep in mind that you have to be careful with constructing the requirements.txt, as missing packages can lead to runtime errors.__

See notebook [30-CreatePipeline](./30-CreatePipeline.ipynb) for how to create the inference wrapper.